In [ ]:
import json
import gradio as gr
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pypdf import PdfReader

# Load environment variables from .env file and override existing ones
load_dotenv(override=True)

# Initialize the OpenAI client following official naming conventions
client = OpenAI()

# Load and parse the LinkedIn resume PDF
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""

# Extract text page by page with defensive null-checks
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

print("--- LinkedIn Resume Content ---")
print(linkedin)

# Read the personal summary text file using UTF-8 encoding
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print("\n--- Personal Summary Content ---")
print(summary)

In [ ]:
# Define the System Prompt containing the Persona, LinkedIn Context, and Rules
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

# Preview the constructed System Prompt in Notebook
display(Markdown(system_prompt))

# -----------------------------------------------------------------------------
# Single Execution Test
# -----------------------------------------------------------------------------
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

response = client.chat.completions.create(
    model="gpt-5.4-mini", 
    messages=messages
)
display(Markdown(response.choices[0].message.content))

In [ ]:
def record_email(email: str) -> str:
    """Record a provided user email address into a local storage file.

    Args:
        email (str): The validated email address string to append.

    Returns:
        str: Operational status response message.
    """
    print(f"Tool Action: Recording email address -> {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email successfully recorded."


# 2. Optimized JSON Schema for OpenAI Tool Calling API
record_email_schema = {
    "name": "record_email",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": (
                    "The valid email address provided by the user, e.g.,"
                    " user@example.com"
                )
            }
        },
        "required": ["email"],
        "additionalProperties": False,
    }
}

# 3. Encapsulate into Tools List for API Ingestion
tools = [{"type": "function", "function": record_email_schema}]

In [ ]:
# -----------------------------------------------------------------------------
# Evaluator / Guardrail Function (LLM Call #2)
# -----------------------------------------------------------------------------
def evaluate_response_relevance(
    user_query: str, primary_response: str
) -> bool:
    """Acts as a post-processing guardrail evaluator using a secondary LLM call

    to verify if the primary response is work-related or legitimate system ops.

    Args:
        user_query (str): The original user prompt.
        primary_response (str): The candidate response generated by LLM 1.

    Returns:
        bool: True if response is professional/work-related or valid system ops,

        False otherwise.
    """
    evaluator_prompt = f"""
You are a strict compliance content evaluator.
Your sole job is to evaluate if the AI's generated response is appropriate for a professional digital twin website.

User Query: "{user_query}"
AI Generated Response: "{primary_response}"

Instructions & Boundary Rules:
- Reply "YES" if the response discusses professional career, skills, work experience, education, OR handles legitimate user actions (such as recording user contact details/emails or thanking the user for contact info).
- Reply "NO" if the response actively discusses off-topic personal hobbies, entertainment, politics, general trivia, or unapproved casual subjects.
- Reply ONLY with "YES" or "NO". Do not include punctuation or extra words.
"""

    eval_response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": evaluator_prompt}],
    )

    result = eval_response.choices[0].message.content.strip().upper()
    return "YES" in result


# -----------------------------------------------------------------------------
# Main Chat Handler with Agent Loop and Short-Circuit Guardrail
# -----------------------------------------------------------------------------
def chat(message: str, history: list) -> str:
    """Handles user interactions via the Agent Loop and applies a Guardrail check

    before outputting the final response.

    Args:
        message (str): Current user input text.
        history (list): Prior conversation history managed by Gradio.

    Returns:
        str: Final validated response string for the UI.
    """
    # Construct base context array
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    # First LLM Call Pass (Primary Agent Reasoning)
    response = client.chat.completions.create(
        model="gpt-5.4-mini", messages=messages, tools=tools
    )

    tool_executed = False

    # Outer While Loop & Inner For Loop for Agent Tool Execution
    while response.choices[0].finish_reason == "tool_calls":
        tool_executed = True
        message_obj = response.choices[0].message
        messages.append(message_obj)

        for tool_call in message_obj.tool_calls:
            args = json.loads(tool_call.function.arguments)
            email = args.get("email")

            # Execute local Python function
            record_email(email)

            # Append tool result matched by tool_call_id
            messages.append(
                {
                    "role": "tool",
                    "content": "Email recorded successfully.",
                    "tool_call_id": tool_call.id,
                }
            )

        # Re-evaluate with updated context window
        response = client.chat.completions.create(
            model="gpt-5.4-mini", messages=messages, tools=tools
        )

    candidate_response = response.choices[0].message.content

    # Short-circuit Mechanism: Bypass Guardrail if a legitimate tool was executed
    if tool_executed:
        return candidate_response

    # Second LLM Call Pass (Guardrail Evaluation for text-only responses)
    is_safe = evaluate_response_relevance(message, candidate_response)

    if is_safe:
        return candidate_response
    else:
        # Fallback response when Guardrail blocks off-topic content
        return (
            "I am designed to discuss topics strictly related to my professional"
            " career, skills, and work experience. How can I help you with"
            " those topics?"
        )


# -----------------------------------------------------------------------------
# Launch Gradio Chat UI
# -----------------------------------------------------------------------------
gr.ChatInterface(chat).launch(inbrowser=True)